# Chapter 5 — VaR Mapping

教材 pp. 60–72；PDF 页序为教材页码加 11。



下列未明确标为教材例题的数据均为合成验证数据，不声称复现教材中的原始市场样本。

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
from numpy.testing import assert_allclose
from mapping_calculator import *

Euler 分解之和等于组合 VaR，基准相同时 tracking VaR 为零；pp.62–65。

In [2]:
positions = pd.DataFrame({'value':[100.,50.]})
cov = pd.DataFrame([[.04,.01],[.01,.09]])
model = LinearRiskMapping(positions,'value',cov)
assert_allclose(model.components().component_var.sum(),model.var())
assert_allclose(model.tracking_var(positions,'value'),0)
print(model.components())

   marginal_var  component_var
0      0.274897      27.489749
1      0.335986      16.799291


现值/久期守恒、远期平价；pp.63,66。

In [3]:
data = pd.DataFrame({'pv':[100.], 'duration':[4.62]})
mapped = duration_map(data,'pv','duration',4,5)
assert_allclose(mapped.lower_exposure,38)
assert_allclose(mapped.upper_exposure,62)
strike = 1.2*np.exp((.04-.02)*2)
assert_allclose(forward_exposures(1.2,strike,.04,.02,2).exposure.sum(),0,atol=1e-12)

## 现金流定价、FRA、swap和delta-gamma；pp.63–72

In [4]:
flows=pd.DataFrame({'cash':[5.,105.],'rate':[.05,.05],'time':[1.,2.]})
pv=cashflow_present_values(flows,'cash','rate','time')
assert_allclose(pv.present_value.sum(),100)
assert_allclose(fra_value(1e6,.04,.03,.5,.97),4850)
swap=swap_cashflows(pd.DataFrame({'accrual':[.5,.5]}),'accrual',100,.04)
assert_allclose(swap.fixed_payment,[-2,-102])
scenarios=delta_gamma_pnl(pd.DataFrame({'change':[-1.,0.,1.]}),'change',.5,.2)
assert_allclose(scenarios.pnl,[-.4,0,.6])